In [ ]:
# Uncomment if running on a fresh Colab runtime
!pip install -q torch-geometric
#!pip install -q torch-scatter torch-sparse torch-cluster torch-spline-conv
# -f https://data.pyg.org/whl/torch-2.8.0+cu128.html

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 44.4 MB/s eta 0:00:00


In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import defaultdict
from torch_geometric.datasets import Planetoid
from torch_geometric.data import Data
from torch_geometric.nn import GATConv
from torch_geometric.utils import degree
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import accuracy_score

In [ ]:
seed = 42
random.seed(seed)
np.random.seed(seed)

torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

cuda


In [ ]:
dataset = Planetoid(
root="./data",
name="Cora"
)
data = dataset[0].to(device)
print("="*60)
print(data)
print("="*60)
print(f"Nodes : {data.num_nodes}")
print(f"Edges : {data.num_edges}")
print(f"Features : {dataset.num_features}")
print(f"Classes : {dataset.num_classes}")

Processing...
Done!


Data(x=[2708, 1433], edge_index=[2, 10556], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708])
Nodes : 2708
Edges : 10556
Features : 1433
Classes : 7


In [ ]:
deg = degree(
data.edge_index[0],
data.num_nodes
).to(device)
print(deg.shape)

torch.Size([2708])


In [ ]:
neighbors = defaultdict(list)
edge_index = data.edge_index.cpu().numpy()
for u, v in zip(edge_index[0], edge_index[1]):
    neighbors[u].append(v)
print(f"Neighbor dictionary built for {len(neighbors)} nodes.")

Neighbor dictionary built for 2708 nodes.


In [ ]:
edge_scores = []

X = data.x.cpu().numpy()

for u, v in zip(edge_index[0], edge_index[1]):

    score = np.dot(X[u], X[v])

    norm = np.linalg.norm(X[u]) * np.linalg.norm(X[v])

    if norm > 0:
        score /= norm
    else:
        score = 0.0

    edge_scores.append((u, v, score))

print(f"Computed affinity for {len(edge_scores)} edges.")

Computed affinity for 10556 edges.


In [ ]:
# ==========================================
# Greedy Affinity Matching
# ==========================================

edge_scores = sorted(edge_scores, key=lambda x: x[2], reverse=True)

matched = np.zeros(data.num_nodes, dtype=bool)

cluster_id = -np.ones(data.num_nodes, dtype=int)

cluster = 0

for u, v, score in edge_scores:

    if matched[u] or matched[v]:
        continue

    matched[u] = True
    matched[v] = True

    cluster_id[u] = cluster
    cluster_id[v] = cluster

    cluster += 1

# Remaining unmatched nodes become singleton clusters
for node in range(data.num_nodes):

    if not matched[node]:

        cluster_id[node] = cluster
        cluster += 1

num_clusters = cluster

print("="*60)
print(f"Original Nodes : {data.num_nodes}")
print(f"Coarse Nodes   : {num_clusters}")
print(f"Compression    : {num_clusters/data.num_nodes:.2f}x")
print("="*60)

Original Nodes : 2708
Coarse Nodes   : 1667
Compression    : 0.62x


In [ ]:
feature_dim = data.x.size(1)

coarse_x = torch.zeros(
    (num_clusters, feature_dim),
    device=device
)

cluster_degree = torch.zeros(
    num_clusters,
    device=device
)

for node in range(data.num_nodes):

    cid = cluster_id[node]

    w = deg[node]

    coarse_x[cid] += w * data.x[node]

    cluster_degree[cid] += w

cluster_degree = cluster_degree.clamp(min=1)

coarse_x /= cluster_degree.unsqueeze(1)

print(coarse_x.shape)

torch.Size([1667, 1433])


In [ ]:
# ==========================================
# Build Coarse Graph
# ==========================================

edge_set = set()

for u, v in zip(edge_index[0], edge_index[1]):

    cu = cluster_id[u]
    cv = cluster_id[v]

    if cu == cv:
        continue

    edge_set.add((cu, cv))
    edge_set.add((cv, cu))

coarse_edge_index = torch.tensor(
    list(edge_set),
    dtype=torch.long
).t().contiguous().to(device)

print(coarse_edge_index.shape)

torch.Size([2, 6986])


In [ ]:
# ==========================================
# Coarse Labels
# ==========================================

coarse_y = torch.zeros(
    num_clusters,
    dtype=torch.long,
    device=device
)

members = [[] for _ in range(num_clusters)]

for node in range(data.num_nodes):
    members[cluster_id[node]].append(node)

for cid in range(num_clusters):

    labels = data.y[members[cid]]

    coarse_y[cid] = torch.mode(labels).values

In [ ]:
import random
import math

# ==========================================
# PURE CLUSTER MASKS (Redistributed based on percentages)
# ==========================================

# Collect all valid cluster IDs first
all_valid_cids = []

for cid in range(num_clusters):
    nodes = members[cid]
    labels = data.y[nodes]

    # Only consider clusters with a single, consistent label
    if labels.unique().numel() != 1:
        continue

    # Check if the cluster belongs to only one original split (train/val/test)
    # This part is crucial to ensure clean splits later for the coarsened graph
    train_mask_for_nodes = data.train_mask[nodes].any().item()
    val_mask_for_nodes = data.val_mask[nodes].any().item()
    test_mask_for_nodes = data.test_mask[nodes].any().item()

    if train_mask_for_nodes + val_mask_for_nodes + test_mask_for_nodes == 1:
        all_valid_cids.append(cid)

# Shuffle all valid cluster IDs for random assignment
random.shuffle(all_valid_cids)

total_valid_clusters = len(all_valid_cids)

# Define target percentages
target_train_ratio = 0.80 # Changed to 80%
target_val_ratio = 0.10  # Kept at 10%

# Calculate target counts based on total valid clusters
target_train_count = math.ceil(total_valid_clusters * target_train_ratio)
target_val_count = math.ceil(total_valid_clusters * target_val_ratio)

# Ensure the sum does not exceed total_valid_clusters and adjust test_count
if target_train_count + target_val_count > total_valid_clusters:
    if target_train_count > total_valid_clusters * target_train_ratio:
        target_train_count = int(total_valid_clusters * target_train_ratio)
    if target_val_count > total_valid_clusters * target_val_ratio:
        target_val_count = int(total_valid_clusters * target_val_ratio)

# Allocate clusters
train_cids_final = all_valid_cids[:target_train_count]
val_cids_final = all_valid_cids[target_train_count : target_train_count + target_val_count]
test_cids_final = all_valid_cids[target_train_count + target_val_count :]

# Recreate coarse_train, coarse_val, coarse_test masks
coarse_train = torch.zeros(num_clusters, dtype=torch.bool, device=device)
coarse_val = torch.zeros_like(coarse_train)
coarse_test = torch.zeros_like(coarse_train)
valid_cluster = torch.zeros_like(coarse_train) # Will mark all clusters used in any split

for cid in train_cids_final:
    coarse_train[cid] = True
    valid_cluster[cid] = True
for cid in val_cids_final:
    coarse_val[cid] = True
    valid_cluster[cid] = True
for cid in test_cids_final:
    coarse_test[cid] = True
    valid_cluster[cid] = True

print("="*60)
print("Pure Clusters (New Redistribution)")
print("="*60)
print("Train :", coarse_train.sum().item())
print("Val   :", coarse_val.sum().item())
print("Test  :", coarse_test.sum().item())

Pure Clusters (New Redistribution)
Train : 752
Val   : 94
Test  : 93


In [ ]:
coarse_data = Data(
x=coarse_x,
edge_index=coarse_edge_index,
y=coarse_y
)
coarse_data.train_mask = coarse_train
coarse_data.val_mask = coarse_val
coarse_data.test_mask = coarse_test
coarse_data = coarse_data.to(device)
print(coarse_data)


Data(x=[1667, 1433], edge_index=[2, 6986], y=[1667], train_mask=[1667], val_mask=[1667], test_mask=[1667])


In [ ]:
class GAT(nn.Module):

    def __init__(self, in_dim, hidden, out_dim):

        super().__init__()

        self.conv1 = GATConv(
            in_dim,
            hidden,
            heads=8,
            dropout=0.6
        )

        self.bn1 = nn.BatchNorm1d(hidden * 8)

        self.conv2 = GATConv(
            hidden * 8,
            hidden,
            heads=8,
            concat=False,
            dropout=0.6
        )

        self.bn2 = nn.BatchNorm1d(hidden)

        self.conv3 = GATConv(
            hidden,
            out_dim,
            heads=1,
            concat=False,
            dropout=0.6
        )

    def forward(self,x,edge_index):

        x1 = self.conv1(x,edge_index)
        x1 = self.bn1(x1)
        x1 = F.elu(x1)

        x2 = self.conv2(x1,edge_index)
        x2 = self.bn2(x2)
        x2 = F.elu(x2)

        x2 = x2 + x1[:, :x2.shape[1]]

        x2 = F.dropout(
            x2,
            p=0.6,
            training=self.training
        )

        out = self.conv3(x2,edge_index)

        return out,x2

In [ ]:
# ==========================================
# Compactness Loss
# ==========================================

cluster_tensor = torch.tensor(
    cluster_id,
    dtype=torch.long,
    device=device
)

def compactness_loss(node_features,
                     coarse_features,
                     cluster_tensor):

    reconstructed = coarse_features[cluster_tensor]

    return F.mse_loss(
        reconstructed,
        node_features
    )

In [ ]:
model = GAT(
    coarse_data.num_features,
    hidden=256,
    out_dim=dataset.num_classes
).to(device)

criterion = nn.CrossEntropyLoss(
    label_smoothing=0.15
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.001,
    weight_decay=4e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=400
)

In [ ]:
best_val = 0
best_test = 0

# ============================================================
# History Containers
# ============================================================

loss_history = []

train_history = []

val_history = []

test_history = []

lr_history = []

for epoch in range(500):

    model.train()

    optimizer.zero_grad()

    logits, embeddings = model(
        coarse_data.x,
        coarse_data.edge_index
    )

    loss_cls = criterion(
        logits[coarse_data.train_mask],
        coarse_data.y[coarse_data.train_mask]
    )

    loss_feat = compactness_loss(
        data.x,
        coarse_data.x,
        cluster_tensor
    )

    loss = (
        loss_cls
        + 0.01 * loss_feat
    )

    loss.backward()

    optimizer.step()

    loss_history.append(loss.item())
    lr_history.append(optimizer.param_groups[0]['lr'])

    ##########################################

    model.eval()

    with torch.no_grad():

        logits, _ = model(
            coarse_data.x,
            coarse_data.edge_index
        )

        pred = logits.argmax(1)

        train_acc = (
            pred[coarse_data.train_mask]
            ==
            coarse_data.y[coarse_data.train_mask]
        ).float().mean()

        val_acc = (
            pred[coarse_data.val_mask]
            ==
            coarse_data.y[coarse_data.val_mask]
        ).float().mean()

        test_acc = (
            pred[coarse_data.test_mask]
            ==
            coarse_data.y[coarse_data.test_mask]
        ).float().mean()

        train_history.append(train_acc.item())
        val_history.append(val_acc.item())
        test_history.append(test_acc.item())

        scheduler.step(val_acc)

        if val_acc > best_val:

            best_val = val_acc
        if test_acc > best_test:
           best_test = test_acc

    if epoch % 10 == 0:

        print(
            f"{epoch:03d} | "
            f"Loss {loss:.4f} | "
            f"Val {val_acc:.4f} | "
            f"Test {test_acc:.4f}"
        )

print("="*70)
print(f"Best Validation : {best_val:.4f}")
print(f"Best Test       : {best_test:.4f}")
print("="*70)

/tmp/ipykernel_1634/1376347774.py:87: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  scheduler.step(val_acc)


000 | Loss 3.4374 | Val 0.5957 | Test 0.6667
010 | Loss 2.0344 | Val 0.7660 | Test 0.8387
020 | Loss 1.6927 | Val 0.7553 | Test 0.7957
030 | Loss 1.6291 | Val 0.8404 | Test 0.8817
040 | Loss 1.4874 | Val 0.8298 | Test 0.9032
050 | Loss 1.4598 | Val 0.8085 | Test 0.9032
060 | Loss 1.3721 | Val 0.8404 | Test 0.9247
070 | Loss 1.3377 | Val 0.8298 | Test 0.9032
080 | Loss 1.3130 | Val 0.8298 | Test 0.8925
090 | Loss 1.2845 | Val 0.8191 | Test 0.8925
100 | Loss 1.2527 | Val 0.8191 | Test 0.8710
110 | Loss 1.1853 | Val 0.8191 | Test 0.8817
120 | Loss 1.1795 | Val 0.8617 | Test 0.8817
130 | Loss 1.2098 | Val 0.8511 | Test 0.8817
140 | Loss 1.1591 | Val 0.8298 | Test 0.8602
150 | Loss 1.1953 | Val 0.8298 | Test 0.8710
160 | Loss 1.1264 | Val 0.8617 | Test 0.8602
170 | Loss 1.1241 | Val 0.8404 | Test 0.8710
180 | Loss 1.1673 | Val 0.8617 | Test 0.8817
190 | Loss 1.1133 | Val 0.8511 | Test 0.8925
200 | Loss 1.1243 | Val 0.8404 | Test 0.8817
210 | Loss 1.1158 | Val 0.8298 | Test 0.8817
220 | Loss